# CS-781 Phase 5 — Legal Statute Identification (notebook I)

Same structure as Phase 4: your Phase-1 models + **your** N6 code (head, full / partial / BitFit / LoRA / LoRMA,
rsLoRA / IA³ — copied unchanged from Phase 4, plus one new function you write: the multi-label distillation loss), a
training script, and a **lab** that turns every finished run into many Codabench zips plus a table and chart of
tentative scores.

**Multi-label.** The head has one output per statute id (sigmoid, not softmax). A training example is a whole
**document**: its windows go through the model, each window's token scores are pooled (`token_pool`), the windows are
pooled into one score per statute (`doc_pool`, e.g. *max*: a statute counts if any window shows it), and a binary
cross-entropy (or focal) loss compares that with the document's statutes.

**Long documents.** Every document is tokenised once (N8, cached) and cut into windows of `window` tokens. At
prediction time every window (up to `score_max_windows`) is scored and saved; the lab decides which windows count
(`SELECTS`: all / head / tail / head+tail — the section comparison the brief asks for), how windows are combined
(`WINDOW_AGGS`), and how scores become a **set** of statute ids (`RULES`: one threshold for all, a threshold **per
statute** fitted on dev, or top-k; `MIN1`: at least one statute per document), then probability ensembles and greedy
ensembles (Caruana et al. 2004).

**Tentative scores** = the official metric (macro-F1 over every statute id 0..max gold id, exactly as
`evaluate_phase5.py`) on the official dev documents. Per-statute thresholds are fitted on dev, so their score is
**cross-fitted** (fit on one half of dev, score the other half, and swap) — otherwise it would be optimistic.
`submissions/index.csv` + `submissions/score_chart.png` list them; **you choose what to upload**. Settings marked YOUR
CHOICE are yours (rule §17.5); declare that the pipeline / lab / chart code was written with an assistant.

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** · Add-ons → Secrets → **`Capstone_Phase_1` attached**.

**Inputs (Add Input):**
| Input | Provides |
|---|---|
| `notebook857159535b` | tokenizer (`bpe_tok.py`, `bpe.json`) |
| `notebook27acca3fc3` (notebook D, phase1-continue) | **decoder** checkpoint `continued/best_continued.pt` |
| `Phase1_Training1` | encdec, swa checkpoints |
| dataset `prefix` | prefix checkpoint |
| `phase1-train-extra` (phase1-training2) | moe (the distillation teacher's base), diffusion |
| *optional:* the output of an earlier run of **this** notebook | its finished runs join the lab; its `p5cache` is reused |

**How to use:** write `kd_loss_multilabel` in N6 (section 7), then Save & Run All (~5 h for the six runs, two at a
time). N7b stops within a minute if an input is missing or the distillation loss is not written yet. Then Output →
`submissions/`: `index.csv`, `score_chart.png` and the zips; download them into your `Phase 5` folder.

## N1. Login + download the Phase 5 data

In [ ]:
import os, shutil, json
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
os.makedirs("/kaggle/working/p5data", exist_ok=True)
for path in ["Train/train.json", "Train/dev.json", "Test/test.json"]:
    src = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_5_LSI/{path}", repo_type="dataset", token=HF_TOKEN)
    shutil.copy(src, f"/kaggle/working/p5data/{os.path.basename(path)}")
print({f: f"{os.path.getsize('/kaggle/working/p5data/' + f) / 1e6:.1f} MB" for f in sorted(os.listdir("/kaggle/working/p5data"))})

## N2. GPU check

In [ ]:
import torch
print("GPU count:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## N3. Configuration — experiments, LSI options, lab settings

In [ ]:
import json
# =====================================================================================================
#  N3 — CONFIGURATION. Values marked YOUR CHOICE are yours (course rule §17.5); the comments explain each one.
# =====================================================================================================
#
#  PHASE 5 RULES THIS CONFIG MUST SATISFY (§11)
#  * all FOUR required Phase-1 architectures used: decoder, prefix (hybrid), encdec, swa (efficient);
#    the best four submissions come from DIFFERENT Phase-1 models
#  * ≥ 4 substantially different experiments: ≥ 3 different techniques FROM THE §11.2.1 LIST — Full fine-tuning,
#    Partial/Layer-wise fine-tuning, QLoRA, LORMA, AdaLoRA, Adapters, Prefix Tuning, Prompt Tuning, Knowledge
#    Distillation — LORMA COMPULSORY — and ≥ 1 knowledge distillation.
#    The starting plan below covers: LoRMA (encdec_lorma), full (prefix_full, moe_full), partial (swa_partial,
#    decoder_partial), distillation (kd_moe_to_decoder4). BitFit / LoRA / rsLoRA / IA³ are NOT on this phase's list.
#  * multi-label: a document can have several statutes -> sigmoid per statute, a set of ids per document.
#  * "handle long documents" — every document is cut into windows; train_chunks + doc_pool decide how the windows
#    are used in training, SELECTS / WINDOW_AGGS compare sections and window pooling at prediction time.

CONFIG = {
    "seed": 42,
    "checkpoints": {
        "decoder":   "continued/best_continued.pt",
        "prefix":    "prefix/best.pt",
        "encdec":    "encdec/best.pt",
        "swa":       "swa/best.pt",
        "diffusion": "diffusion/best.pt",
        "moe":       "moe/best.pt",
    },

    # ---------------- defaults for every experiment (an experiment below can override any of them) ----------------
    "defaults": {
        # --- LSI-specific: which windows of a document the model learns from, and how they become one prediction ---
        "train_chunks": "all",   # windows of each document used per TRAINING step (a window = `window` tokens):
                                 #   "all"      a random sample of at most max_windows_per_case windows (new each epoch)
                                 #   "head" / "tail" / "headtail"   first / last / first + last window
        "max_windows_per_case": 4,   # YOUR CHOICE. More windows = more of each document seen, more GPU time per step
        "batch_windows": 8,      # windows per training step (documents are packed up to this); 8 x 1024 as in Phase 4
        "token_pool": "mean",    # YOUR CHOICE. a window's statute scores = "mean" / "max" over its tokens, or its
                                 #   "last" token (causal models: the only token that has seen the whole window)
        "doc_pool": "max",       # YOUR CHOICE. a document's scores from its windows' scores: "max" (a statute counts if
                                 #   one window shows it), "mean", or "lse" (log-mean-exp, a smooth max)
        "loss": "bce",           # YOUR CHOICE. "bce" (binary cross-entropy per statute) or "focal" (down-weights easy
        "focal_gamma": 2.0,      #   negatives; Lin et al. 2017) with this gamma
        "pos_weight": "none",    # YOUR CHOICE. rare statutes: weight each statute's positives by negatives/positives
        "pos_weight_cap": 30.0,  #   ("balanced") or its square root ("sqrt"), capped here; "none" = no weighting
        # --- compute: documents are long and many, so each run has a time budget ---
        "time_budget_min": 75,   # YOUR CHOICE. Training stops after this many minutes (or after `epochs`), then the
                                 # best checkpoint scores dev + test (~10-25 min more). 6 runs on two GPUs ≈ 5 h.
        "eval_minutes": 25,      # evaluate every this many minutes (and at each epoch end) on eval_docs dev documents;
        "eval_docs": 1500,       # the best evaluation is the checkpoint that is kept (the final scoring uses all of dev)
        "score_max_windows": 16, # YOUR CHOICE. windows scored per document at prediction time (first 8 + last 8 when a
                                 # document has more); None = all (slower). N8 prints how many windows documents have.
        "epochs": 3,             # upper limit on passes over the training documents
        # --- the rest as in Phases 2-4 ---
        "window": 1024,          # tokens per window (capped at what the model accepts: encdec 768)
        "context": "plain",
        "lr": 5e-5,              # YOUR CHOICE
        "warmup_frac": 0.1,
        "weight_decay": 0.01,
        "grad_clip": 1.0,
        "head_dropout": 0.1,
        "method_cfg": {},
        "moe_aux_coef": 0.0,
        "train_on_all": False,   # True = train on train + dev (no dev scores)
        "epochs_from": None,     # for train_on_all runs: a finished dev-scored run whose best epoch is reused
        "teacher": None,
        "kd": {"temperature": 2.0, "alpha": 0.5},
        "student_layers": None,
    },

    # ---------------- the experiments — YOUR CHOICE (starting plan = your Phase-4 settings) ----------------
    "experiments": {
        "moe_full":           {"arch": "moe", "method": "full"},                       # also the distillation teacher
        "kd_moe_to_decoder4": {"arch": "decoder", "method": "full", "teacher": "moe_full", "student_layers": 4},
        "encdec_lorma":       {"arch": "encdec", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "prefix_full":        {"arch": "prefix", "method": "full"},
        "swa_partial":        {"arch": "swa", "method": "partial"},      # your method_partial (default n_last=2)
        "decoder_partial":    {"arch": "decoder", "method": "partial"},
        # --- optional extras (not in SESSION_EXPERIMENTS below): ---
        "swa_bitfit":         {"arch": "swa", "method": "bitfit", "lr": 1e-3},
        "decoder_lora":       {"arch": "decoder", "method": "lora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "diffusion_full":     {"arch": "diffusion", "method": "full"},
        # "name_focal": {<settings of another run>, "loss": "focal", "pos_weight": "sqrt"},  <- loss / class balance
        # "name_ht":    {<settings of another run>, "train_chunks": "headtail"},           <- section comparison
        # "name_all":   {<settings of a dev-scored run>, "train_on_all": True, "epochs_from": "<that run>"},
    },
}

# ---------------- which experiments THIS session trains (GPU) and smoke-tests (CPU) ----------------
SESSION_EXPERIMENTS = ["moe_full", "prefix_full", "encdec_lorma", "swa_partial", "kd_moe_to_decoder4", "decoder_partial"]
SMOKE_EXPERIMENTS = list(SESSION_EXPERIMENTS)             # a distillation student waits for its teacher

# ---------------- lab: post-processing + ensembles of finished runs (N11) — YOUR CHOICE ----------------
SELECTS = ["all", "head", "tail", "headtail"]            # parts of the document used at prediction time
WINDOW_AGGS = ["max", "mean", "lse"]                     # how window scores become document scores
RULES = ["thr0.2", "thr0.3", "thr0.4", "thr0.5",         # thrX: statute predicted when P >= X (same X for all)
         "perlabel",                                     # one threshold per statute, fitted on dev (cross-fitted score)
         "top1", "top2", "top3"]                         # topK: the K most probable statutes
MIN1 = [True, False]                                     # True: a document with no statute gets its most probable one
PROB_ENSEMBLES = {}          # {"name": ["run_a", "run_b"]} -> every variant, dev-scored;
                             # {"name": {"members": [...], "select": "all", "window_agg": "max", "rule": "thr0.3", "min1": True}}
GREEDY = True
RUN_DIR_OVERRIDES = {}
TOP_N_ZIPS = 40              # zips for the 40 best sets (+ the best set of every run / ensemble)
MAX_LABEL = None             # safety: never predict a statute id above this (the official evaluator REJECTS a whole
                             # submission containing an id > the largest id in its test reference). None = largest
                             # id seen in train + dev. N8 prints the largest ids of train and dev.

json.dump(CONFIG, open("/kaggle/working/run_config_ls.json", "w"), indent=1)
for name, e in CONFIG["experiments"].items():
    assert e["arch"] in CONFIG["checkpoints"], f"{name}: unknown arch {e['arch']}"
    t = e.get("teacher")
    assert not t or t in CONFIG["experiments"], f"{name}: teacher {t} is not an experiment"
    f = e.get("epochs_from")
    assert not f or (e.get("train_on_all") and f in CONFIG["experiments"]), f"{name}: epochs_from needs train_on_all and a known run"
for name in SESSION_EXPERIMENTS:
    assert name in CONFIG["experiments"], f"SESSION_EXPERIMENTS: {name} is not defined"
n = len(SESSION_EXPERIMENTS)
budget = [CONFIG["experiments"][e].get("time_budget_min", CONFIG["defaults"]["time_budget_min"]) for e in SESSION_EXPERIMENTS]
print(f"{len(CONFIG['experiments'])} experiments defined; this session trains {n}: {SESSION_EXPERIMENTS}")
if n:
    print(f"rough GPU time: ~{sum(b + 20 for b in budget) / 2 / 60:.1f} h on T4 x2 (time budgets + scoring)")

## N4. Pipeline code: data + label space, tokenisation cache, windows, pooling, metrics, backbones, files

In [ ]:
%%writefile ls_common.py
"""Phase 5 (Legal Statute Identification) pipeline: data + label space, one-time Phase-1 tokenisation of every document
(cached on disk), windows of a long document, window-level statute scores, document pooling, metrics (same rules as
evaluate_phase5.py), Phase-1 backbones that return hidden states, run finder, files."""
import glob, json, os, sys
from multiprocessing import Pool
import numpy as np, torch, torch.nn as nn

INPUT_ROOT = os.environ.get("P5_INPUT", "/kaggle/input")
WORK = os.environ.get("P5_WORK", "/kaggle/working")
DATA = f"{WORK}/p5data"
CACHE = f"{WORK}/p5cache"
SPLITS = ("train", "dev", "test")
LEN_BUCKET = 32          # a window shorter than `window` is cut to a multiple of this (fewer distinct lengths = bigger batches)

def find_input(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{pattern}' not found under {INPUT_ROOT} — is the right notebook/dataset added as input?")
    return hits[0]

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

_BPE = None
def bpe_tok_bpe():
    global _BPE
    if _BPE is None:
        _BPE = bpe_tok.load_bpe(find_input("bpe.json"))
    return _BPE

# ------------------------------------------------------------------------------------------------ data
def _labels(x):
    """a document's statute ids -> sorted list of unique ints"""
    if isinstance(x, (int, np.integer)) and not isinstance(x, bool):
        x = [x]
    if isinstance(x, str):
        x = [s for s in x.replace(";", ",").split(",") if s.strip()]
    out = set()
    for v in x:
        if isinstance(v, bool):
            raise ValueError(f"statute id must be an integer, got {v!r}")
        if isinstance(v, str):
            if not v.strip().lstrip("-").isdigit():
                raise ValueError(f"statute id must be an integer, got {v!r}")
            v = int(v.strip())
        if not isinstance(v, (int, np.integer)) or v < 0:
            raise ValueError(f"statute id must be a non-negative integer, got {v!r}")
        out.add(int(v))
    return sorted(out)

def read_split(name):
    """-> list of {"id": str, "text": str, "labels": [int, ...] or None}. Accepts the documented list of
    {"id", "text", "labels"} and close variants (a dict keyed by id; "label" / "gold" / "statutes" instead of
    "labels"; text given as a list of sentences)."""
    obj = json.load(open(f"{DATA}/{name}.json", encoding="utf-8"))
    if isinstance(obj, dict) and isinstance(obj.get("data"), list):
        obj = obj["data"]
    items = list(obj.items()) if isinstance(obj, dict) else [(None, x) for x in obj]
    out = []
    for k, it in items:
        if isinstance(it, str):
            it = {"text": it}
        doc_id = it.get("id", it.get("case_id", it.get("doc_id", k)))
        text = it.get("text", it.get("document", it.get("facts")))
        if isinstance(text, list):
            text = " ".join(map(str, text))
        if doc_id is None or not isinstance(text, str):
            raise ValueError(f"{name}.json: cannot read an example (keys: {list(it)[:8]}). Expected "
                             '{"id": ..., "text": "...", "labels": [1, 7, 23]}')
        lab = next((it[key] for key in ("labels", "label", "gold", "statutes", "y") if key in it), None)
        out.append({"id": str(doc_id), "text": text, "labels": None if lab is None else _labels(lab)})
    return out

# ------------------------------------------------------------------------------------------------ tokenisation cache
_ENC = None
def _init_worker():
    global _ENC
    _ENC = bpe_tok.FastEncoder(bpe_tok_bpe())

def _encode(text):
    return np.asarray(_ENC.encode(text), dtype=np.int32)

def build_cache(workers=None, limit=None):
    """Tokenises every document ONCE with your Phase-1 tokenizer (bpe_tok.FastEncoder, as in pretraining) and stores
    per split: all token ids concatenated (ids.npy) + where each document starts (offsets.npy) + ids/labels (meta.json).
    Training runs then read these files instead of re-tokenising."""
    os.makedirs(CACHE, exist_ok=True)
    for split in SPLITS:
        if os.path.exists(f"{CACHE}/{split}_meta.json"):
            continue
        docs = read_split(split)[:limit]
        with Pool(workers or os.cpu_count(), initializer=_init_worker) as pool:
            arrs = pool.map(_encode, [d["text"] for d in docs], chunksize=16)
        offsets = np.zeros(len(arrs) + 1, dtype=np.int64)
        offsets[1:] = np.cumsum([len(a) for a in arrs])
        dtype = np.uint16 if len(bpe_tok_bpe().itos) <= 65535 else np.int32
        np.save(f"{CACHE}/{split}_ids.npy", np.concatenate(arrs).astype(dtype) if arrs else np.zeros(0, dtype))
        np.save(f"{CACHE}/{split}_offsets.npy", offsets)
        json.dump({"ids": [d["id"] for d in docs], "labels": [d["labels"] for d in docs]}, open(f"{CACHE}/{split}_meta.json", "w"))
        print(f"{split}: {len(docs)} documents, {offsets[-1]:,} tokens cached")

class Split:
    """one cached split: ids, label lists, and token ids of document i via split.tokens(i) (memory-mapped)"""
    def __init__(self, name):
        m = json.load(open(f"{CACHE}/{name}_meta.json"))
        self.name, self.ids, self.labels = name, m["ids"], m["labels"]
        self.flat = np.load(f"{CACHE}/{name}_ids.npy", mmap_mode="r")
        self.off = np.load(f"{CACHE}/{name}_offsets.npy")
    def __len__(self):
        return len(self.ids)
    def tokens(self, i):
        t = np.asarray(self.flat[self.off[i]:self.off[i + 1]], dtype=np.int32)
        return t if len(t) else np.array([bpe_tok_bpe().stoi["<unk>"]], dtype=np.int32)
    def lengths(self):
        return np.diff(self.off)

def n_labels():
    """size of the classification head = largest statute id in train + dev, plus 1 (ids are used as column indices)"""
    m = max(max(l) for s in ("train", "dev") for l in Split(s).labels if l)
    return m + 1

def multi_hot(labels, k):
    y = np.zeros(k, dtype=np.float32)
    y[[x for x in labels if x < k]] = 1.0
    return y

# ------------------------------------------------------------------------------------------------ windows of a document
def doc_windows(n, win):
    """non-overlapping windows covering the document; the last one is aligned to the end (so it is the last `win`
    tokens). First window = head, last window = tail. A document shorter than `win` is one window, cut to a multiple
    of LEN_BUCKET tokens (your models have no padding mask, so a batch needs equal lengths)."""
    if n <= win:
        return [(0, n if n < LEN_BUCKET else n - n % LEN_BUCKET)]
    starts = list(range(0, n - win, win)) + [n - win]
    return [(s, s + win) for s in sorted(set(starts))]

def capped_windows(n, win, cap):
    """at most `cap` windows (None = all): the first ceil(cap/2) and the last floor(cap/2), in document order"""
    w = doc_windows(n, win)
    if cap and len(w) > cap:
        w = w[:(cap + 1) // 2] + w[len(w) - cap // 2:]
    return w

SELECTS = ("all", "head", "tail", "headtail")

def select_windows(n_rows, how):
    """which stored windows of a document count. all | head (first) | tail (last) | headtail (first + last)"""
    if how == "all" or n_rows == 1:
        return list(range(n_rows))
    return {"head": [0], "tail": [n_rows - 1], "headtail": [0, n_rows - 1]}[how]

def train_windows_for(n, win, how, max_windows, rng):
    """windows a document contributes to one training step: head / tail / headtail, or all (a random sample of at
    most max_windows of them)"""
    wins = doc_windows(n, win)
    if how == "all":
        idx = list(range(len(wins)))
        if max_windows and len(idx) > max_windows:
            idx = sorted(rng.choice(len(idx), max_windows, replace=False).tolist())
    else:
        idx = select_windows(len(wins), how)
    return [wins[i] for i in idx]

def doc_batches(order, n_windows, batch_windows):
    """groups documents (in `order`) into steps of about batch_windows windows (always at least one document)"""
    out, cur, used = [], [], 0
    for i in order:
        if cur and used + n_windows[i] > batch_windows:
            out.append(cur); cur, used = [], 0
        cur.append(i); used += n_windows[i]
    if cur:
        out.append(cur)
    return out

# ------------------------------------------------------------------------------------------------ pooling
TOKEN_POOLS = ("mean", "last", "max")
WINDOW_AGGS = ("max", "mean", "lse")

def pool_tokens(logits, how):
    """(n, T, K) token logits of n windows -> (n, K) window logits"""
    if how == "mean":
        return logits.mean(1)
    if how == "last":
        return logits[:, -1]
    if how == "max":
        return logits.max(1).values
    raise ValueError(f"token_pool must be one of {TOKEN_POOLS}, got {how!r}")

def pool_windows(z, how):
    """(n_windows, K) window logits of ONE document -> (K,) document logits. Works on torch tensors and numpy arrays.
    max: a statute counts if one window shows it · mean: average evidence · lse: log-mean-exp, a smooth max"""
    if how == "max":
        return z.max(0).values if torch.is_tensor(z) else z.max(0)
    if how == "mean":
        return z.mean(0)
    if how == "lse":
        if torch.is_tensor(z):
            return torch.logsumexp(z, 0) - float(np.log(z.shape[0]))
        m = z.max(0)
        return m + np.log(np.exp(z - m).mean(0))
    raise ValueError(f"doc_pool must be one of {WINDOW_AGGS}, got {how!r}")

def window_logits(model, seqs, token_pool, device):
    """forward a list of equal-or-different-length token arrays in groups of equal length -> (len(seqs), K) tensor"""
    by_len = {}
    for j, s in enumerate(seqs):
        by_len.setdefault(len(s), []).append(j)
    out = [None] * len(seqs)
    for _, js in by_len.items():
        x = torch.tensor(np.stack([seqs[j] for j in js]), dtype=torch.long, device=device)
        z = pool_tokens(model(x).float(), token_pool)
        for j, row in zip(js, z):
            out[j] = row
    return torch.stack(out)

@torch.no_grad()
def window_scores(model, split, win, device, token_pool, batch_windows=8, max_seqs=256, idx=None, cap=None):
    """For every document: one row per window (in document order) = the window's statute logits pooled over its
    tokens as in training. -> {document id: (n_windows, K) float16}"""
    idx = range(len(split)) if idx is None else idx
    jobs, n_win = [], {}
    for i in idx:
        t = split.tokens(i)
        for w, (s, e) in enumerate(capped_windows(len(t), win, cap)):
            jobs.append((i, w, t[s:e]))
        n_win[i] = w + 1
    out = {}
    by_len = {}
    for j in jobs:
        by_len.setdefault(len(j[2]), []).append(j)
    for L_, group in by_len.items():
        k = max(1, min(max_seqs, (batch_windows * win) // L_))
        for a in range(0, len(group), k):
            chunk = group[a:a + k]
            x = torch.tensor(np.stack([c[2] for c in chunk]), dtype=torch.long, device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):
                z = pool_tokens(model(x).float(), token_pool).float().cpu().numpy()
            for (i, w, _), row in zip(chunk, z):
                if i not in out:
                    out[i] = np.zeros((n_win[i], len(row)), dtype=np.float32)
                out[i][w] = row
    return {split.ids[i]: np.clip(out[i], -60000, 60000).astype(np.float16) for i in sorted(out)}   # repeated id: last wins

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.asarray(x, dtype=np.float32)))

def doc_probs(rows, window_agg="max", select="all"):
    """P(statute k applies) for every k, from a document's window rows"""
    rows = np.asarray(rows, dtype=np.float32)
    return sigmoid(pool_windows(rows[select_windows(len(rows), select)], window_agg))

# ------------------------------------------------------------------------------------------------ metrics
def label_matrix(label_lists, k):
    y = np.zeros((len(label_lists), k), dtype=bool)
    for r, ls in enumerate(label_lists):
        y[r, [x for x in ls if x < k]] = True
    return y

def per_label_f1(gold, pred):
    """gold, pred: (N, n) bool -> F1 of every column; 0 when a column has no gold and no prediction
    (sklearn zero_division=0, as in evaluate_phase5.py)"""
    tp = (gold & pred).sum(0); fp = (~gold & pred).sum(0); fn = (gold & ~pred).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

def scores(gold, pred):
    """official Phase-5 metrics on (N, n) bool matrices, n = largest GOLD statute id + 1 (columns of pred beyond n are
    invalid for the official evaluator and are ignored here). macro_f1 = mean F1 over ALL n columns."""
    n = gold.shape[1]
    pred = pred[:, :n]
    tp = (gold & pred).sum(); fp = (~gold & pred).sum(); fn = (gold & ~pred).sum()
    return {"macro_f1": float(per_label_f1(gold, pred).mean()),
            "micro_f1": float(2 * tp / (2 * tp + fp + fn)) if 2 * tp + fp + fn else 0.0,
            "subset_acc": float((gold == pred).all(1).mean()),
            "micro_p": float(tp / (tp + fp)) if tp + fp else 0.0,
            "micro_r": float(tp / (tp + fn)) if tp + fn else 0.0}

# ------------------------------------------------------------------------------------------------ backbones
TINY = {   # used only by the CPU smoke test (random weights, no checkpoint)
    "decoder":   dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "prefix":    dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "encdec":    dict(d_model=32, n_enc_layers=2, n_dec_layers=1, n_heads=4, max_len=256, max_tgt_len=32, dropout=0.0),
    "swa":       dict(d_model=32, n_layers=2, n_heads=4, window=16, max_len=256, dropout=0.0),
    "diffusion": dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "moe":       dict(d_model=32, n_layers=2, n_heads=4, d_ff=64, n_experts=4, max_len=256, dropout=0.0),
}

class Backbone(nn.Module):
    """Wraps one of YOUR Phase-1 models so that backbone(ids) returns hidden states (B, T, d_model) instead of
    next-token scores (output layer replaced by Identity). encdec: the ENCODER output is used.
    context="echo" (causal models): the window is fed twice and the second copy's states are used (Springer et al. 2024)."""
    def __init__(self, arch, lm, bos_id, context="plain"):
        super().__init__()
        self.arch, self.lm, self.bos_id, self.context = arch, lm, bos_id, context
        self.kind = getattr(lm, "kind", "causal")
        self.d_model = lm.tok_emb.embedding_dim
        self.max_input = lm.max_len // 2 if context == "echo" else lm.max_len
        lm.head = nn.Identity()
        if arch == "encdec":
            del lm.dec_layers, lm.dec_norm, lm.tgt_pos_emb

    def forward(self, ids):
        T0 = ids.size(1)
        x = torch.cat([ids, ids], 1) if self.context == "echo" else ids
        T = x.size(1)
        if self.arch == "encdec":
            lm = self.lm
            h = lm.tok_emb(x) + lm.src_pos_emb(torch.arange(T, device=x.device))
            for layer in lm.enc_layers:
                h = layer(h)
            h = lm.enc_norm(h)
        elif self.kind == "prefix":
            h = self.lm(x, prefix_len=T)
        else:
            h = self.lm(x)
        return h[:, T - T0:] if self.context == "echo" else h

def build_lm(arch, cfg):
    import models, models_x
    return (models_x if arch in ("diffusion", "moe") else models).build_model(arch, vocab_size=len(bpe_tok_bpe().itos), **cfg)

def load_backbone(arch, ckpt_pattern, context="plain", smoke=False):
    bpe = bpe_tok_bpe()
    if smoke:
        lm = build_lm(arch, TINY[arch])
    else:
        ck = torch.load(find_input(ckpt_pattern), map_location="cpu", weights_only=False)
        assert ck.get("arch", arch) == arch, f"checkpoint {ckpt_pattern} is a '{ck.get('arch')}' model, not '{arch}'"
        lm = build_lm(arch, ck["cfg"])
        lm.load_state_dict(ck["model"])
    return Backbone(arch, lm, bpe.stoi["<s>"], context)

def iter_adaptable(module, skip=("head",)):
    for name, mod in module.named_modules():
        if any(s in name.split(".") for s in skip):
            continue
        if isinstance(mod, nn.MultiheadAttention) and getattr(mod, "in_proj_weight", None) is not None:
            yield name, mod, "in_proj_weight", mod.in_proj_weight
        elif isinstance(mod, nn.Linear):
            yield name, mod, "weight", mod.weight

def layer_stack(backbone):
    lm = backbone.lm
    if backbone.arch == "encdec":
        return "enc_layers", lm.enc_layers, lm.enc_norm
    if hasattr(lm, "blocks"):
        return "blocks", lm.blocks, lm.norm
    return "layers", lm.layers, lm.norm

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    return total, sum(p.numel() for p in model.parameters() if p.requires_grad)

# ------------------------------------------------------------------------------------------------ runs + files
def find_runs(filename, overrides=None):
    """{experiment: folder} for every '.../ls_runs/<experiment>/<filename>' under /kaggle/working and /kaggle/input.
    Found more than once: /kaggle/working wins, then the newest file."""
    hits = {}
    for rank, root in enumerate((WORK, INPUT_ROOT)):
        for p in glob.glob(f"{root}/**/{filename}", recursive=True):
            folder = os.path.dirname(p)
            if os.path.basename(os.path.dirname(folder)) != "ls_runs":
                continue
            exp, key = os.path.basename(folder), (rank, -os.path.getmtime(p))
            if exp not in hits or key < hits[exp][0]:
                hits[exp] = (key, folder)
    out = {e: f for e, (_, f) in sorted(hits.items())}
    out.update(overrides or {})
    return out

def write_submission(ids, preds, path_json, path_zip=None):
    """official format: [{"id": "...", "pred": [statute ids]}, ...], one entry per test document, in test order;
    ids are unique ints, sorted"""
    json.dump([{"id": i, "pred": sorted({int(x) for x in preds[i]})} for i in ids], open(path_json, "w", encoding="utf-8"))
    if path_zip:
        import zipfile
        with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(path_json, arcname="prediction.json")

## N5. Your Phase-1 model code (unchanged copies of your `models.py` and `models_x.py`)

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

In [ ]:
%%writefile models_x.py

# =====================================================================================================
# CS-781 Phase 1 — two extra architectures
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
# PART 0 — HELPERS
# =====================================================================================================

def causal_mask(T, device):
    return torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    ).triu(diagonal=1)


def make_layers(n_layers, d_model, n_heads, dropout):
    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(
            module.weight,
            mean=0.0,
            std=0.02
        )

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
# MODEL 5 — LexDiffusionLM
# =====================================================================================================

class LexDiffusionLM(nn.Module):
    kind = "diffusion"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer layers
        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Fully bidirectional attention.
        # NO MASK.
        for layer in self.layers:
            x = layer(x)

        # Final normalization + output
        x = self.norm(x)

        return self.head(x)


# =====================================================================================================
# MODEL 6 — LexMoE
# =====================================================================================================

class MoEFeedForward(nn.Module):
    """
    Top-1 Mixture-of-Experts feed-forward network.

    Each token is routed to exactly one expert.
    """

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        self.n_experts = n_experts

        # Filled during forward()
        self.aux_loss = None
        self.last_choice = None

        # Router
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False
        )

        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):
        B, T, D = x.shape

        # Flatten tokens
        flat = x.reshape(
            B * T,
            D
        )

        # Router probabilities
        router_logits = self.router(flat)

        # Float32 router softmax for numerical stability
        probs = torch.softmax(
            router_logits.float(),
            dim=-1
        )

        # Best expert for each token
        top_p, top_i = probs.max(
            dim=-1
        )

        # Output buffer
        out = flat.new_zeros(
            flat.shape
        )

        # ------------------------------------------------
        # Route tokens to experts
        # ------------------------------------------------

        for e in range(self.n_experts):

            sel = (top_i == e)

            if sel.any():

                expert_out = self.experts[e](
                    flat[sel]
                )

                expert_out = (
                    expert_out
                    * top_p[sel][:, None]
                )

                out[sel] = expert_out.to(
                    out.dtype
                )


        # ------------------------------------------------
        # Switch load-balancing loss
        # ------------------------------------------------

        counts = torch.bincount(
            top_i,
            minlength=self.n_experts
        ).float()

        frac_e = counts / flat.shape[0]

        prob_e = probs.mean(
            dim=0
        )

        self.aux_loss = (
            self.n_experts
            * torch.sum(frac_e * prob_e)
        )


        # Store routing decisions
        self.last_choice = top_i.reshape(
            B,
            T
        )


        # Restore original shape
        return out.reshape(
            B,
            T,
            D
        )


# =====================================================================================================
# MoE BLOCK
# =====================================================================================================

class MoEBlock(nn.Module):
    """
    Pre-norm Transformer block:

        x -> LayerNorm -> Attention -> residual
          -> LayerNorm -> MoE      -> residual
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        # Pre-norm attention
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.drop = nn.Dropout(
            dropout
        )

        # Pre-norm MoE
        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.moe = MoEFeedForward(
            d_model,
            d_ff,
            n_experts,
            dropout
        )


    def forward(self, x, mask):

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + self.drop(
            attn_out
        )


        # ------------------------------------------------
        # MoE feed-forward
        # ------------------------------------------------

        moe_out = self.moe(
            self.norm2(x)
        )

        x = x + moe_out

        return x


# =====================================================================================================
# LexMoE
# =====================================================================================================

class LexMoE(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        n_experts,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Filled during forward()
        self.aux_loss = None

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # MoE Transformer blocks
        self.blocks = nn.ModuleList([
            MoEBlock(
                d_model,
                n_heads,
                d_ff,
                n_experts,
                dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Causal mask
        mask = causal_mask(
            T,
            idx.device
        )

        # Store auxiliary losses
        aux_losses = []

        # Transformer blocks
        for block in self.blocks:

            x = block(
                x,
                mask
            )

            aux_losses.append(
                block.moe.aux_loss
            )

        # Average auxiliary loss across layers
        self.aux_loss = torch.stack(
            aux_losses
        ).mean()

        # Final normalization
        x = self.norm(x)

        # Vocabulary logits
        return self.head(x)


# =====================================================================================================
# FACTORY — DO NOT CHANGE
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "diffusion": LexDiffusionLM,
        "moe": LexMoE
    }

    return classes[arch](
        vocab_size,
        **cfg
    )

## N6. `ls_methods.py` — YOUR CODE (unchanged from Phase 4) + one new function to write
The head (`TokenClassifier`) works as-is: it gets one output per statute. Full, partial and LoRMA are on the §11.2.1
list and already written. **New this phase — section 7 at the bottom:** `kd_loss_multilabel`, the distillation loss
for multi-label outputs (your `kd_loss` is softmax / single-label). The steps are written out there; N7b stops the
session if it is still missing and the distillation run is scheduled.

In [ ]:
%%writefile ls_methods.py
# =====================================================================================================
#  CS-781 Phase 5 (copied from Phase 4) — YOUR CODE: the NER head, the fine-tuning methods, and knowledge distillation
#                                                                AUTHOR: <your name>   (your own code)
# =====================================================================================================

import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
import ls_common as L


# =====================================================================================================
#  1. THE NER HEAD
# =====================================================================================================

class TokenClassifier(nn.Module):

    def __init__(self, backbone, n_tags, dropout):
        super().__init__()

        # Keep backbone
        self.backbone = backbone

        # Dropout before classifier
        self.dropout = nn.Dropout(dropout)

        # Token-level classification head
        self.head = nn.Linear(
            backbone.d_model,
            n_tags
        )


    def forward(self, ids):

        # Backbone hidden states
        h = self.backbone(ids)

        # Classification
        return self.head(
            self.dropout(h)
        )


# =====================================================================================================
#  2. SIMPLE METHODS: full, partial, BitFit
# =====================================================================================================

def method_full(model, **cfg):
    """Full fine-tuning: every parameter of the backbone trains."""

    for p in model.backbone.parameters():
        p.requires_grad_(True)


def method_partial(model, n_last=2, **cfg):
    """
    Partial / layer-wise fine-tuning:
    only the LAST n_last transformer layers + final LayerNorm train.
    """

    # Get the relevant layer stack
    name, layers, final_norm = L.layer_stack(
        model.backbone
    )

    # Train only the last n_last layers
    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad_(True)

    # Train final LayerNorm
    for p in final_norm.parameters():
        p.requires_grad_(True)


def method_bitfit(model, **cfg):
    """
    BitFit:
    only bias parameters of the backbone train.
    """

    for name, p in model.backbone.named_parameters():

        if name.endswith("bias"):
            p.requires_grad_(True)


# =====================================================================================================
#  3. LoRA
# =====================================================================================================

class LoRAParam(nn.Module):
    """
    Additive low-rank update:

        W' = W + (alpha / r) * B @ A
    """

    def __init__(
        self,
        d_out,
        d_in,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.empty(r, d_in)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5)
        )

        # B: zero initialization
        # This guarantees W' = W initially.
        self.B = nn.Parameter(
            torch.zeros(d_out, r)
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        return W + self.scale * (
            self.B @ self.A
        )


def method_lora(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRAParam(
                d_out,
                d_in,
                r,
                alpha
            )
        )


# =====================================================================================================
#  4. LoRMA
# =====================================================================================================

class LoRMAParam(nn.Module):
    """
    LoRMA+ multiplicative low-rank update:

        W' = W + (alpha / r) * B @ (A @ W)

    where:

        A : (r, d_out)
        B : (d_out, r)
    """

    def __init__(
        self,
        d_out,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.randn(
                r,
                d_out
            ) * 0.02
        )

        # B: zero initialization
        self.B = nn.Parameter(
            torch.zeros(
                d_out,
                r
            )
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        # First compute A @ W
        AW = self.A @ W

        # Then B @ (A @ W)
        update = self.B @ AW

        return W + self.scale * update


def method_lorma(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRMAParam(
                d_out,
                r,
                alpha
            )
        )


# =====================================================================================================
#  5. KNOWLEDGE DISTILLATION
# =====================================================================================================

def kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):
    """
    Combined hard-label cross entropy and
    teacher-student KL divergence.
    """

    # Only real labelled tokens
    mask = labels != -100

    s = student_logits[mask]
    t = teacher_logits[mask]
    y = labels[mask]

    # --------------------------------------------------------
    # Hard-label loss
    # --------------------------------------------------------

    hard = F.cross_entropy(
        s,
        y
    )

    # --------------------------------------------------------
    # Soft teacher/student distributions
    # --------------------------------------------------------

    soft = F.kl_div(
        F.log_softmax(
            s / temperature,
            dim=-1
        ),
        F.softmax(
            t / temperature,
            dim=-1
        ),
        reduction="batchmean"
    )

    # Hinton temperature scaling
    soft = soft * (
        temperature * temperature
    )

    # Combined loss
    return (
        alpha * hard
        + (1.0 - alpha) * soft
    )


def make_student(
    backbone,
    keep_layers
):
    """
    Make a smaller copy of a Phase-1 backbone,
    keeping only its first keep_layers layers.
    """

    # Deep copy so original backbone is untouched
    small = copy.deepcopy(
        backbone
    )

    # Find the layer stack
    name, layers, _ = L.layer_stack(
        small
    )

    # Keep only first keep_layers
    setattr(
        small.lm,
        name,
        nn.ModuleList(
            list(layers)[:keep_layers]
        )
    )

    return small


# =====================================================================================================
#  METHOD TABLE
# =====================================================================================================

METHODS = {
    "full": method_full,
    "partial": method_partial,
    "bitfit": method_bitfit,
    "lora": method_lora,
    "lorma": method_lorma,
}


# =====================================================================================================
#  6. YOUR PHASE-3 ADDITIONS (rsLoRA, IA³) — unchanged. Note for Phase 4: the §10.2.1 list is Full fine-tuning,
#     Partial/Layer-wise fine-tuning, Layer freezing/unfreezing, BitFit, LORMA, AdaLoRA, Adapters, Knowledge
#     Distillation. rsLoRA, IA³ and LoRA are NOT on it, so they are optional extras this phase. You already have
#     full, partial (method_partial above), BitFit, LoRMA and distillation — no new method is needed.
# =====================================================================================================

def method_rslora(model, r=8, alpha=16, skip=("head",), **cfg):
    """Rank-stabilised LoRA: use alpha / sqrt(r) as the update scale."""
    if r <= 0:
        raise ValueError("r must be a positive integer")

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out, d_in = W.shape
        adapter = LoRAParam(d_out, d_in, r, alpha)
        adapter.scale = alpha / math.sqrt(r)
        parametrize.register_parametrization(owner, pname, adapter)


# -----------------------------------------------------------------------------------------------------
#  B. IA³ — "Infused Adapter by Inhibiting and Amplifying Inner Activations" (Liu et al. 2022)
# -----------------------------------------------------------------------------------------------------
#  In plain English:
#    IA³ adds no matrices at all. Every adapted weight gets ONE learned vector, with one number per output
#    row of the weight. Each row of the weight is multiplied by its number. The vector starts as all ones, so
#    at the start the model is unchanged (the same idea as B starting at zero in LoRA). Only these vectors and
#    the head are trained, so IA³ trains far fewer parameters than LoRA.
#
#  What you have to write — two pieces:
#
#    Piece 1 — a small class, written just above method_ia3, shaped like your LoRAParam:
#      1. It inherits from nn.Module and its __init__ receives d_out (the number of output rows of the weight).
#      2. In __init__, after super().__init__(), create one trainable vector of length d_out filled with ones
#         (an nn.Parameter wrapped around torch.ones of size d_out). Call it something like "l".
#      3. Its forward receives the original weight W (shape d_out x d_in) and returns W with every row multiplied
#         by the matching number of the vector. To do that, turn the vector into a column of shape d_out x 1
#         (index it with [:, None]) and multiply it with W — PyTorch repeats the column across all d_in columns.
#
#    Piece 2 — the function method_ia3:
#      4. Go through every adaptable weight with the same loop as method_lora.
#      5. Read the weight's number of output rows (the first number of W.shape).
#      6. Create your Piece-1 object with that number and attach it with parametrize.register_parametrization,
#         exactly like method_lora attaches LoRAParam.
#      7. Delete the "raise NotImplementedError" line.
#
#    For your report: the paper rescales only keys, values and the feed-forward hidden layer. Rescaling every
#    adaptable weight (what steps 4-6 do) is a simpler variant — say which one you used.

class IA3Param(nn.Module):
    """Row-wise IA³ scaling for a weight matrix."""

    def __init__(self, d_out):
        super().__init__()
        self.l = nn.Parameter(torch.ones(d_out))

    def forward(self, W):
        return self.l[:, None] * W


def method_ia3(model, skip=("head",), **cfg):
    """Apply trainable row-wise scaling to every adaptable backbone weight."""
    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out = W.shape[0]
        parametrize.register_parametrization(
            owner,
            pname,
            IA3Param(d_out),
        )


# Do not change this line: it makes "rslora" and "ia3" usable as "method" in N3.
METHODS.update({"rslora": method_rslora, "ia3": method_ia3})


# =====================================================================================================
#  7. NEW FOR PHASE 5 — distillation loss for MULTI-LABEL outputs. YOU write this function (about 6 lines).
# =====================================================================================================
#  Why a new one: your kd_loss above is for single-label classification — a softmax over the classes, and
#  cross_entropy against ONE class index per token. In LSI every statute is its own yes/no decision (a document can
#  have several), so the teacher's knowledge is K independent probabilities, one sigmoid per statute.
#
#  What train_ls.py passes in (already pooled to ONE row per document):
#    student_logits  (B, K)  the student's statute scores        teacher_logits (B, K)  the teacher's
#    targets         (B, K)  the true statutes, 1.0 / 0.0         temperature, alpha     from "kd" in N3
#
#  Steps:
#    1. hard loss: F.binary_cross_entropy_with_logits(student_logits, targets)
#    2. the teacher's soft targets: torch.sigmoid(teacher_logits / temperature)
#    3. soft loss: F.binary_cross_entropy_with_logits(student_logits / temperature, <the soft targets of step 2>),
#       multiplied by temperature * temperature (the same Hinton scaling as in your kd_loss)
#    4. return alpha * hard + (1 - alpha) * soft
#    5. delete the "raise NotImplementedError" line.
#  For your report: this is the per-statute (binary) form of Hinton et al.'s distillation.

def kd_loss_multilabel(student_logits, teacher_logits, targets, temperature=2.0, alpha=0.5):
    hard = F.binary_cross_entropy_with_logits(student_logits, targets)
    soft_targets = torch.sigmoid(teacher_logits.detach() / temperature)
    soft = F.binary_cross_entropy_with_logits(
        student_logits / temperature, soft_targets
    ) * (temperature * temperature)
    return alpha * hard + (1.0 - alpha) * soft

## N7. Scripts: training (one experiment), the lab, the official Phase-5 evaluator

In [ ]:
%%writefile train_ls.py
"""Runs ONE experiment from run_config_ls.json: your Phase-1 backbone + your head (TokenClassifier with one output per
statute) + your fine-tuning method. A training example is a DOCUMENT: its windows (train_chunks) go through the model,
each window's token scores are pooled (token_pool), the windows are pooled into one score per statute (doc_pool), and
the loss compares that with the document's statutes (multi-label). Evaluates on dev every eval_minutes, keeps the best
checkpoint (dev macro-F1), stops at time_budget_min or after `epochs`. Writes the window scores of every dev / test
document (dev_scores.pkl, test_scores.pkl) for the lab (N11), plus its own dev / test predictions and Codabench zip.
   python train_ls.py --exp NAME --device cuda:0        (real run)
   python train_ls.py --exp NAME --device cpu --smoke   (tiny random model, few documents: checks the code)"""
import argparse, json, os, pickle, random, time
import numpy as np, torch, torch.nn.functional as F
import ls_common as L
import ls_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
cuda = dev.startswith("cuda")
if cuda:
    torch.cuda.set_device(dev)

C = json.load(open(f"{L.WORK}/run_config_ls.json"))
def exp_cfg(name):
    if name in C["experiments"]:
        return {**C["defaults"], **C["experiments"][name], "name": name}
    raise KeyError(f"experiment '{name}' is not in EXPERIMENTS (cell N3)")
E = exp_cfg(args.exp)
RUNS = f"{L.WORK}/{'ls_runs_smoke' if args.smoke else 'ls_runs'}"
EVAL_THRESHOLDS = (0.1, 0.2, 0.3, 0.4, 0.5)        # dev evaluation: macro-F1 at the best of these (one for all statutes)

def best_epoch_of(name):
    p = f"{RUNS}/{name}/summary.json"
    if not os.path.exists(p):
        p = f"{L.find_runs('summary.json')[name]}/summary.json"
    return json.load(open(p))["best_epoch"]

if E["train_on_all"] and E.get("epochs_from") and not args.smoke:
    E["epochs"] = best_epoch_of(E["epochs_from"])         # may be fractional (an epoch is cut by the time budget)
if args.smoke:
    E.update(window=96, epochs=1, batch_windows=4, max_windows_per_case=2, time_budget_min=3, eval_minutes=99,
             eval_docs=4, score_max_windows=3)
OUT = f"{RUNS}/{args.exp}"; os.makedirs(OUT, exist_ok=True)
log = open(f"{OUT}/log.jsonl", "w")
t0 = time.time()
def say(**kw):
    kw["t"] = round(time.time() - t0); log.write(json.dumps(kw) + "\n"); log.flush()
    print(args.exp, json.dumps(kw), flush=True)
seed = E.get("seed", C["seed"]); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------------------------------------------ data (tokenised once in N8)
train, dsplit, test = L.Split("train"), L.Split("dev"), L.Split("test")
K = L.n_labels()
train_idx = [i for i in range(len(train)) if train.labels[i]]
dev_idx = list(range(len(dsplit)))
test_idx = list(range(len(test)))
if args.smoke:
    train_idx, dev_idx, test_idx = train_idx[:8], dev_idx[:4], test_idx[:4]
dev_gold = {dsplit.ids[i]: dsplit.labels[i] for i in dev_idx}   # a repeated id is counted once (its last occurrence)
dev_ids = list(dev_gold)
n_dev = max(max(l) for l in dsplit.labels if l) + 1             # label dimension the official evaluator would use
G_dev = L.label_matrix([dev_gold[i] for i in dev_ids], n_dev)
sources = [(train, i) for i in train_idx] + ([(dsplit, i) for i in dev_idx if dsplit.labels[i]] if E["train_on_all"] else [])
rng = np.random.RandomState(seed)
eval_ids = dev_ids if len(dev_ids) <= E["eval_docs"] else sorted(rng.choice(dev_ids, E["eval_docs"], replace=False).tolist())
row_of = {i: r for r, i in enumerate(dev_ids)}
last_of = {dsplit.ids[j]: j for j in dev_idx}
eval_rows = [row_of[i] for i in eval_ids]
eval_idx = sorted(last_of[i] for i in eval_ids)

# ------------------------------------------------------------------ model
def build(e, smoke):
    backbone = L.load_backbone(e["arch"], C["checkpoints"][e["arch"]], context=e["context"], smoke=smoke)
    if e.get("student_layers"):
        backbone = M.make_student(backbone, e["student_layers"])
    model = M.TokenClassifier(backbone, n_tags=K, dropout=e["head_dropout"])
    for p in model.parameters():
        p.requires_grad_(False)
    M.METHODS[e["method"]](model, **e["method_cfg"])
    for p in model.head.parameters():
        p.requires_grad_(True)
    return model

model = build(E, args.smoke).to(dev)
win = min(E["window"], model.backbone.max_input)
teacher, TE = None, None
if E.get("teacher"):
    TE = exp_cfg(E["teacher"])
    tpath = f"{RUNS}/{E['teacher']}/best.pt"
    if not os.path.exists(tpath):
        tpath = f"{L.find_runs('best.pt')[E['teacher']]}/best.pt"
    teacher = build(TE, args.smoke)
    teacher.load_state_dict(torch.load(tpath, map_location="cpu", weights_only=False)["state"])
    teacher.to(dev).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    win = min(win, teacher.backbone.max_input)
total, trainable = L.count_params(model)
label_lists = [s.labels[i] for s, i in sources]
pos = np.bincount([x for ls in label_lists for x in ls], minlength=K)[:K].astype(np.float64)
say(event="start", arch=E["arch"], method=E["method"], params_total=total, params_trainable=trainable,
    trainable_pct=round(100 * trainable / total, 3), window=win, n_labels=K, train_chunks=E["train_chunks"],
    token_pool=E["token_pool"], doc_pool=E["doc_pool"], loss=E["loss"], pos_weight=E["pos_weight"],
    train_docs=len(sources), labels_per_doc=round(float(np.mean([len(x) for x in label_lists])), 2),
    dev_docs=len(dev_ids), time_budget_min=E["time_budget_min"], teacher=E.get("teacher"))

# ------------------------------------------------------------------ training setup
pw = None
if E["pos_weight"] != "none":                 # weight of a statute's positives = its negatives / positives (balanced)
    ratio = (len(label_lists) - pos) / np.maximum(pos, 1)   # or the square root of that (sqrt), capped
    w = np.sqrt(ratio) if E["pos_weight"] == "sqrt" else ratio
    w = np.where(pos > 0, np.minimum(w, E["pos_weight_cap"]), 1.0)
    pw = torch.tensor(w, dtype=torch.float32, device=dev)

def task_loss(z, y):
    """z: (B, K) document logits, y: (B, K) multi-hot. bce, or focal (Lin et al. 2017) on top of it"""
    ce = F.binary_cross_entropy_with_logits(z, y, pos_weight=pw, reduction="none")
    if E["loss"] == "focal":
        p = torch.sigmoid(z)
        p_t = p * y + (1 - p) * (1 - y)
        ce = (1 - p_t).pow(E["focal_gamma"]) * ce
    elif E["loss"] != "bce":
        raise ValueError(f"loss must be 'bce' or 'focal', got {E['loss']!r}")
    return ce.mean()

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=E["lr"], weight_decay=E["weight_decay"])
assert E["max_windows_per_case"] <= E["batch_windows"], "max_windows_per_case must be <= batch_windows"

def n_train_windows(s, i):
    n = len(L.doc_windows(int(s.off[i + 1] - s.off[i]) or 1, win))
    return {"all": min(n, E["max_windows_per_case"]), "headtail": min(n, 2)}.get(E["train_chunks"], 1)

counts = [n_train_windows(s, i) for s, i in sources]
steps_per_epoch = len(L.doc_batches(range(len(sources)), counts, E["batch_windows"]))
budget_s = E["time_budget_min"] * 60
total_steps = max(1, int(steps_per_epoch * E["epochs"]))
warm = max(1, int(E["warmup_frac"] * min(total_steps, steps_per_epoch)))
def lr_at(step):            # warm-up, then linear decay to 10% over the planned steps or the time budget, whichever ends first
    frac = max(step / total_steps, (time.time() - t0) / budget_s)
    return E["lr"] * ((step + 1) / warm if step < warm else max(0.1, 1.0 - frac))
scaler = torch.amp.GradScaler("cuda", enabled=cuda)

def doc_logits(net, cfg, seqs, owner, n_docs):
    """windows -> window logits (token_pool) -> document logits (doc_pool), (n_docs, K)"""
    z = L.window_logits(net, seqs, cfg["token_pool"], dev)
    return torch.stack([L.pool_windows(z[[j for j, o in enumerate(owner) if o == d]], cfg["doc_pool"]) for d in range(n_docs)])

def predict(sc, ids, thr, min1=False):
    P = np.stack([L.doc_probs(sc[i], E["doc_pool"]) for i in ids])
    Y = P >= thr
    if min1:
        Y[np.arange(len(P)), P.argmax(1)] = True
    return P, Y

def dev_eval(idx, ids, rows):
    model.eval()
    sc = L.window_scores(model, dsplit, win, dev, E["token_pool"], E["batch_windows"], idx=idx, cap=E["score_max_windows"])
    model.train()
    best_s, best_t = None, None
    for thr in EVAL_THRESHOLDS:
        s = L.scores(G_dev[rows], predict(sc, ids, thr)[1])
        if best_s is None or s["macro_f1"] > best_s["macro_f1"]:
            best_s, best_t = s, thr
    return sc, best_s, best_t

# ------------------------------------------------------------------ train
best, best_epoch, step, last_eval, stop = -1.0, 0.0, 0, time.time(), False
model.train()
if cuda:
    torch.cuda.reset_peak_memory_stats()
epoch = 0
while not stop:
    epoch += 1
    losses = []
    order = rng.permutation(len(sources))
    for bi, batch in enumerate(L.doc_batches(order, counts, E["batch_windows"])):
        if (args.smoke and bi >= 4) or step >= total_steps or time.time() - t0 > budget_s:
            stop = True
            break
        seqs, owner = [], []
        for d, j in enumerate(batch):
            s, i = sources[j]
            t = s.tokens(i)
            for a, b in L.train_windows_for(len(t), win, E["train_chunks"], E["max_windows_per_case"], rng):
                seqs.append(t[a:b]); owner.append(d)
        y = torch.tensor(np.stack([L.multi_hot(sources[j][0].labels[sources[j][1]], K) for j in batch]), device=dev)
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
            z = doc_logits(model, E, seqs, owner, len(batch)).float()
            if teacher is not None:
                with torch.no_grad():
                    tz = doc_logits(teacher, TE, seqs, owner, len(batch)).float()
                loss = M.kd_loss_multilabel(z, tz, y, **E["kd"])
            else:
                loss = task_loss(z, y)
            aux = getattr(model.backbone.lm, "aux_loss", None)
            if E["moe_aux_coef"] and aux is not None:
                loss = loss + E["moe_aux_coef"] * aux
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, E["grad_clip"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        losses.append(loss.item()); step += 1
        if not E["train_on_all"] and time.time() - last_eval > E["eval_minutes"] * 60:
            _, s, thr = dev_eval(eval_idx, eval_ids, eval_rows); last_eval = time.time()
            ep = round(step / steps_per_epoch, 3)
            say(event="eval", epoch=ep, step=step, train_loss=round(float(np.mean(losses[-200:])), 4),
                dev_macro_f1=round(s["macro_f1"], 4), dev_micro_f1=round(s["micro_f1"], 4), threshold=thr, eval_docs=len(eval_ids))
            if s["macro_f1"] > best:
                best, best_epoch = s["macro_f1"], ep
                torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt")
    if step >= total_steps:
        stop = True
    ep = round(step / steps_per_epoch, 3)
    if losses:
        say(event="epoch_end" if not stop else "train_end", epoch=ep, step=step, train_loss=round(float(np.mean(losses)), 4))
    if E["train_on_all"]:
        torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt"); best_epoch = ep
    else:
        _, s, thr = dev_eval(eval_idx, eval_ids, eval_rows); last_eval = time.time()
        say(event="eval", epoch=ep, step=step, dev_macro_f1=round(s["macro_f1"], 4), dev_micro_f1=round(s["micro_f1"], 4),
            threshold=thr, eval_docs=len(eval_ids))
        if s["macro_f1"] > best:
            best, best_epoch = s["macro_f1"], ep
            torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt")

# ------------------------------------------------------------------ final scores with the best checkpoint
model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=dev, weights_only=False)["state"])
model.eval()
peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else None
summary = {"exp": args.exp, "arch": E["arch"], "method": E["method"], "teacher": E.get("teacher"),
           "student_layers": E.get("student_layers"), "train_chunks": E["train_chunks"], "token_pool": E["token_pool"],
           "doc_pool": E["doc_pool"], "loss": E["loss"], "pos_weight": E["pos_weight"], "seed": seed,
           "train_on_all": E["train_on_all"], "n_labels": K, "best_epoch": best_epoch,
           "epochs_done": round(step / steps_per_epoch, 3), "steps": step, "params_total": total,
           "params_trainable": trainable, "peak_gpu_gb": peak, "window": win, "train_minutes": round((time.time() - t0) / 60, 1)}
thr = 0.5
if not E["train_on_all"]:
    ti = time.time()
    dsc, s, thr = dev_eval(dev_idx, dev_ids, list(range(len(dev_ids))))
    pickle.dump(dsc, open(f"{OUT}/dev_scores.pkl", "wb"))
    summary.update(dev_macro_f1=round(s["macro_f1"], 4), dev_micro_f1=round(s["micro_f1"], 4),
                   dev_subset_acc=round(s["subset_acc"], 4), dev_threshold=thr, dev_predict_sec=round(time.time() - ti, 1))
ti = time.time()
tsc = L.window_scores(model, test, win, dev, E["token_pool"], E["batch_windows"], idx=test_idx, cap=E["score_max_windows"])
pickle.dump(tsc, open(f"{OUT}/test_scores.pkl", "wb"))
summary["test_predict_sec"] = round(time.time() - ti, 1)
tids = [test.ids[i] for i in test_idx]
_, Y = predict(tsc, tids, thr, min1=True)
L.write_submission(tids, {i: np.flatnonzero(y).tolist() for i, y in zip(tids, Y)},
                   f"{OUT}/prediction.json", f"{OUT}/submission_{args.exp}.zip")
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
say(event="done", **summary)

In [ ]:
%%writefile ls_lab.py
"""Phase 5 lab — no training. From the window scores every finished run saved (dev_scores.pkl, test_scores.pkl) it
builds prediction sets: which part of the document is used (SELECTS: all windows / head / tail / head+tail) x how the
windows are combined (WINDOW_AGGS: max / mean / log-mean-exp of their statute scores) x how scores become a SET of
statute ids (RULES: one threshold for all statutes, a threshold per statute fitted on dev, or the top-k statutes)
x whether every document gets at least one statute (MIN1); probability ensembles; greedy ensembles.

Tentative score = the official metric (macro-F1 over every statute id 0..max gold id) on the official dev documents.
For "perlabel" the thresholds are fitted on dev, so its tentative score is CROSS-FITTED: fitted on one half of dev,
scored on the other half, and the other way round (the in-sample score is the fit_f1 column, for reference only); its
test zip uses thresholds fitted on all of dev. Writes Codabench zips, submissions/index.csv and a score chart. You
choose which zips to upload."""
import csv, glob, hashlib, json, os, pickle
import numpy as np
import ls_common as L

KIND_COLORS = {"single": "#2a78d6", "ensemble": "#eb6834", "greedy": "#1baf7a"}   # fixed categorical order
GRID = np.round(np.arange(0.05, 0.951, 0.05), 2)                                 # candidate per-statute thresholds


def rule_name(rule):
    kind, v = rule
    return {"thr": f"thr{v}", "perlabel": "perlabel", "topk": f"top{v}"}[kind]


def parse_rules(rules):
    """["thr0.3", "perlabel", "top2", ...] -> [("thr", 0.3), ("perlabel", None), ("topk", 2), ...]"""
    out = []
    for r in rules:
        if r == "perlabel":
            out.append(("perlabel", None))
        elif r.startswith("thr"):
            out.append(("thr", float(r[3:])))
        elif r.startswith("top"):
            out.append(("topk", int(r[3:])))
        else:
            raise ValueError(f"unknown rule {r!r}: use 'thrX', 'perlabel' or 'topK'")
    return out


class Lab:
    def __init__(self, out_dir=f"{L.WORK}/submissions", max_label=None):
        self.dev, self.test = L.Split("dev"), L.Split("test")
        # dev.json may repeat ids: each id is scored ONCE (its last occurrence), like the official evaluator
        self.gold = {i: g for i, g in zip(self.dev.ids, self.dev.labels) if g}
        self.dev_ids = list(self.gold)
        self.n_dev = max(max(g) for g in self.gold.values()) + 1     # label dimension the official evaluator uses
        self.G = L.label_matrix([self.gold[i] for i in self.dev_ids], self.n_dev)
        self.K = L.n_labels()
        # statute ids that may be predicted: seen in train or dev (never a gap id), and <= max_label if you set one
        seen = np.zeros(self.K, dtype=bool)
        for s in ("train", "dev"):
            for ls in L.Split(s).labels:
                seen[[x for x in (ls or []) if x < self.K]] = True
        if max_label is not None:
            seen[max_label + 1:] = False
        self.valid = seen
        self.fold = np.random.RandomState(0).rand(len(self.dev_ids)) < 0.5   # the two halves for cross-fitting
        self.out_dir = out_dir
        os.makedirs(f"{out_dir}/dev_predictions", exist_ok=True)
        json.dump([{"id": i, "gold": g} for i, g in self.gold.items()], open(f"{out_dir}/dev_predictions/dev_truth.json", "w"))
        self.rows, self.models, self.cache = [], {}, {}

    def load_runs(self, overrides=None):
        for e, f in L.find_runs("test_scores.pkl", overrides).items():
            s = json.load(open(f"{f}/summary.json")) if os.path.exists(f"{f}/summary.json") else {}
            dv = f"{f}/dev_scores.pkl"
            self.models[e] = {"test": pickle.load(open(f"{f}/test_scores.pkl", "rb")),
                              "dev": pickle.load(open(dv, "rb")) if os.path.exists(dv) and not s.get("train_on_all") else None,
                              "doc_pool": s.get("doc_pool", "max"), "folder": f}
        print(f"runs found: {len(self.models)}  |  dev documents {len(self.dev_ids)}, label dimension on dev {self.n_dev}, "
              f"head size {self.K}, predictable statute ids {int(self.valid.sum())}")
        for e, m in self.models.items():
            print(f"  {e:28s} trained doc_pool {m['doc_pool']:5s} {'' if m['dev'] is not None else '(no dev scores: train_on_all)'} <- {m['folder']}")

    # ------------------------------------------------------------------ scores -> probabilities -> statute sets
    def probs(self, members, split, agg, select):
        """(N, K) P(statute applies), averaged over members (probability ensemble); invalid ids set to 0.
        Only single runs are cached (an ensemble is a cheap mean of cached members)."""
        ids = self.dev_ids if split == "dev" else self.test.ids
        P = np.mean([self._member_probs(m, split, agg, select, ids) for m in members], 0)
        P[:, ~self.valid] = 0.0
        return P

    def _member_probs(self, m, split, agg, select, ids):
        key = (m, split, agg, select)
        if key not in self.cache:
            sc = self.models[m][split]
            self.cache[key] = np.stack([L.doc_probs(sc[i], agg, select) for i in ids])
        return self.cache[key]

    def fit_thresholds(self, P, G):
        """one threshold per statute from GRID, maximising that statute's F1 on (P, G); a statute with no positive in
        G, or F1 0 at every threshold, gets the single threshold that maximises macro-F1"""
        n = G.shape[1]
        f1 = np.stack([L.per_label_f1(G, P[:, :n] >= g) for g in GRID])          # (len(GRID), n)
        glob_t = GRID[int(np.argmax(f1.mean(1)))]
        t = np.full(P.shape[1], glob_t)
        ok = (G.sum(0) > 0) & (f1.max(0) > 0)
        t[:n][ok] = GRID[np.argmax(f1[:, ok], 0)]
        return t

    def decide(self, P, rule, min1, thresholds=None):
        kind, v = rule
        if kind == "topk":
            Y = np.zeros(P.shape, dtype=bool)
            np.put_along_axis(Y, np.argsort(-P, 1)[:, :v], True, 1)
        else:
            Y = P >= (v if kind == "thr" else thresholds)
        if min1:
            empty = ~Y.any(1)
            Y[empty, P[empty].argmax(1)] = True
        return Y & self.valid

    def predict(self, members, agg, select, rule, min1, with_dev=True):
        """-> (tentative dev sets, in-sample dev sets, test sets); dev ones are None when a member has no dev scores"""
        Pt = self.probs(members, "test", agg, select)
        if not with_dev or any(self.models[m]["dev"] is None for m in members):
            if rule[0] == "perlabel":
                return None
            return None, None, self.decide(Pt, rule, min1)
        Pd = self.probs(members, "dev", agg, select)
        if rule[0] != "perlabel":
            Yd = self.decide(Pd, rule, min1)
            return Yd, Yd, self.decide(Pt, rule, min1)
        t_all = self.fit_thresholds(Pd, self.G)
        Ycv = np.zeros(Pd.shape, dtype=bool)
        for f in (True, False):                                                  # cross-fitting on the two halves
            fit, ev = self.fold == f, self.fold != f
            Ycv[ev] = self.decide(Pd[ev], rule, min1, self.fit_thresholds(Pd[fit], self.G[fit]))
        return Ycv, self.decide(Pd, rule, min1, t_all), self.decide(Pt, rule, min1, t_all)

    def tentative(self, Yd):
        return L.scores(self.G, Yd)["macro_f1"]

    # ------------------------------------------------------------------ prediction sets
    def add(self, name, kind, res, **info):
        if res is None:
            return None
        Ycv, Yfit, Yt = res
        s = L.scores(self.G, Ycv) if Ycv is not None else None
        r = lambda k: None if s is None else round(s[k], 4)
        row = {"name": name, "kind": kind, "tentative_f1": r("macro_f1"), "micro_f1": r("micro_f1"),
               "subset_acc": r("subset_acc"), "micro_p": r("micro_p"), "micro_r": r("micro_r"),
               "fit_f1": None if Yfit is None else round(L.scores(self.G, Yfit)["macro_f1"], 4),
               "dev_labels_per_doc": None if Ycv is None else round(float(Ycv.sum(1).mean()), 2),
               "test_labels_per_doc": round(float(Yt.sum(1).mean()), 2), **info}
        self.rows.append((row, Ycv, Yt))
        return row["tentative_f1"]

    def variants(self, name, kind, members, selects, aggs, rules, min1s):
        for agg in aggs:
            for sel in selects:
                for rule in rules:
                    for m1 in min1s:
                        self.add(f"{name}__{sel}__{agg}__{rule_name(rule)}{'__min1' if m1 else ''}", kind,
                                 self.predict(members, agg, sel, rule, m1), members="+".join(members),
                                 select=sel, window_agg=agg, rule=rule_name(rule), min1=m1)

    def singles(self, selects, aggs, rules, min1s):
        for e, m in self.models.items():
            if m["dev"] is None:
                print(f"{e}: no dev scores (train_on_all) -> its zip is the one train_ls.py wrote; use it in PROB_ENSEMBLES")
                continue
            self.variants(e, "single", [e], selects, aggs, rules, min1s)

    def ensembles(self, groups, selects, aggs, rules, min1s):
        """{name: [members]} -> every variant, dev-scored; {name: {"members", "select", "window_agg", "rule", "min1"}}
        -> that one variant (for train_on_all members, which have no dev scores: rule thrX or topK)"""
        for name, g in groups.items():
            members = g["members"] if isinstance(g, dict) else list(g)
            missing = [m for m in members if m not in self.models]
            if missing:
                print(f"ensemble {name}: skipped, no scores for {missing}")
                continue
            if not isinstance(g, dict):
                self.variants(f"ens_{name}", "ensemble", members, selects, aggs, rules, min1s)
                continue
            rule = parse_rules([g["rule"]])[0]
            res = self.predict(members, g["window_agg"], g["select"], rule, g["min1"])
            if res is None:
                print(f"ensemble {name}: 'perlabel' needs dev scores of every member — skipped")
                continue
            self.add(f"ens_{name}__{g['select']}__{g['window_agg']}__{g['rule']}{'__min1' if g['min1'] else ''}",
                     "ensemble", res, members="+".join(members), select=g["select"], window_agg=g["window_agg"],
                     rule=g["rule"], min1=g["min1"])

    def greedy(self, selects, aggs, rules, min1s):
        """Greedy forward ensemble selection on dev (Caruana et al. 2004), one per window aggregation: repeatedly add
        the run that raises the tentative macro-F1 the most (best select x rule x min1), stop when nothing helps."""
        for agg in aggs:
            pool = [e for e, m in self.models.items() if m["dev"] is not None]
            def objective(members):
                best = (-1.0, None, None, None)
                for sel in selects:
                    for rule in rules:
                        for m1 in min1s:
                            f = self.tentative(self.predict(members, agg, sel, rule, m1)[0])
                            if f > best[0]:
                                best = (f, sel, rule, m1)
                return best
            chosen, best = [], (-1.0, None, None, None)
            while True:
                tries = [(objective(chosen + [e]), e) for e in pool if e not in chosen]
                if not tries:
                    break
                res, e = max(tries, key=lambda x: x[0][0])
                if res[0] <= best[0]:
                    break
                chosen.append(e); best = res
            if len(chosen) < 2:
                print(f"greedy {agg}: no ensemble beats the best single run ({chosen})")
                continue
            f, sel, rule, m1 = best
            self.add(f"greedy__{sel}__{agg}__{rule_name(rule)}{'__min1' if m1 else ''}", "greedy",
                     self.predict(chosen, agg, sel, rule, m1), members="+".join(chosen), select=sel, window_agg=agg,
                     rule=rule_name(rule), min1=m1)
            print(f"greedy {agg}: {chosen} -> tentative macro-F1 {f:.4f} ({sel}, {rule_name(rule)}, min1={m1})")

    # ------------------------------------------------------------------ files
    def finish(self, top_n=30):
        """zips for: the top_n sets by tentative macro-F1, the best set of each run / ensemble, every set without dev
        scores. Identical test predictions are written once. index.csv lists every set."""
        scored = sorted([r for r in self.rows if r[0]["tentative_f1"] is not None], key=lambda r: -r[0]["tentative_f1"])
        keep = {id(r) for r in scored[:top_n]}
        best_of = {}
        for r in scored:
            best_of.setdefault(r[0].get("members"), r)
        keep |= {id(r) for r in best_of.values()} | {id(r) for r in self.rows if r[0]["tentative_f1"] is None}
        order = scored + [r for r in self.rows if r[0]["tentative_f1"] is None]
        for f in glob.glob(f"{self.out_dir}/*.zip") + glob.glob(f"{self.out_dir}/dev_predictions/*__*.json"):
            os.remove(f)
        seen, n = {}, 0
        tid = self.test.ids
        for r in order:
            row, Yd, Yt = r
            row["file"] = ""
            if id(r) not in keep:
                continue
            h = hashlib.md5(np.packbits(Yt, axis=1).tobytes()).hexdigest()
            if h in seen:
                row["file"] = f"(same as {seen[h]})"
                continue
            seen[h] = row["file"] = f"{row['name']}.zip"
            L.write_submission(tid, {i: np.flatnonzero(y).tolist() for i, y in zip(tid, Yt)},
                               f"{self.out_dir}/_prediction.json", f"{self.out_dir}/{row['file']}")
            if Yd is not None:      # dev predictions in the official format (ids beyond the dev label dimension dropped)
                json.dump([{"id": i, "pred": [int(x) for x in np.flatnonzero(y) if x < self.n_dev]}
                           for i, y in zip(self.dev_ids, Yd)], open(f"{self.out_dir}/dev_predictions/{row['name']}.json", "w"))
            n += 1
        if os.path.exists(f"{self.out_dir}/_prediction.json"):
            os.remove(f"{self.out_dir}/_prediction.json")
        cols = ["name", "kind", "tentative_f1", "micro_f1", "subset_acc", "micro_p", "micro_r", "fit_f1",
                "dev_labels_per_doc", "test_labels_per_doc", "members", "select", "window_agg", "rule", "min1", "file"]
        with open(f"{self.out_dir}/index.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
            w.writeheader()
            for row, _, _ in order:
                w.writerow(row)
        print(f"{len(self.rows)} prediction sets scored, {n} zips written to {self.out_dir}/ (index.csv lists all)")
        return [r[0] for r in order]


def table(rows, n=25):
    scored = [r for r in rows if r["tentative_f1"] is not None]
    print(f"{'macroF1':>7s} {'microF1':>7s} {'subset':>6s} {'P':>6s} {'R':>6s} {'#/doc':>5s}  {'prediction set':62s} zip")
    for r in scored[:n]:
        print(f"{r['tentative_f1']:7.4f} {r['micro_f1']:7.4f} {r['subset_acc']:6.3f} {r['micro_p']:6.3f} {r['micro_r']:6.3f} "
              f"{r['test_labels_per_doc']:5.2f}  {r['name'][:62]:62s} {r.get('file', '')}")
    for r in rows:
        if r["tentative_f1"] is None and r.get("file"):
            print(f"{'n/a':>7s} {'':7s} {'':6s} {'':6s} {'':6s} {r['test_labels_per_doc']:5.2f}  {r['name'][:62]:62s} {r['file']}")


def chart(rows, path, n=25):
    """Tentative score chart (macro-F1, the official metric, on the official dev documents): left = the n best
    prediction sets coloured by kind; right = the best set of every run / ensemble with macro-F1, micro-F1 and subset
    accuracy. index.csv is the table view of the same numbers."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    scored = [r for r in rows if r["tentative_f1"] is not None]
    if not scored:
        print("no dev scores to plot"); return
    top = scored[:n][::-1]
    best_of = {}
    for r in scored:
        best_of.setdefault(r["members"], r)
    groups = list(best_of.values())[:15][::-1]
    ink, muted, grid = "#0b0b0b", "#52514e", "#e4e3df"
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(17, max(6, 0.32 * max(len(top), 3 * len(groups) / 1.6))),
                                 gridspec_kw={"width_ratios": [1.15, 1]})
    fig.patch.set_facecolor("#fcfcfb")
    for ax in (a1, a2):
        ax.set_facecolor("#fcfcfb")
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.spines["bottom"].set_color(grid)
        ax.tick_params(colors=muted, labelsize=8, length=0)
        ax.grid(axis="x", color=grid, linewidth=0.8)
        ax.set_axisbelow(True)
    y = np.arange(len(top))
    a1.barh(y, [r["tentative_f1"] for r in top], height=0.62, color=[KIND_COLORS[r["kind"]] for r in top],
            edgecolor="#fcfcfb", linewidth=2)
    a1.set_yticks(y, [r["name"][:62] for r in top], fontsize=7.5, color=ink)
    for yi, r in zip(y, top):
        a1.text(r["tentative_f1"] + 0.003, yi, f"{r['tentative_f1']:.3f}", va="center", fontsize=7.5, color=muted)
    a1.set_xlim(0, min(1.0, max(r["tentative_f1"] for r in top) * 1.12))
    a1.set_xlabel("tentative macro-F1 on dev (per-statute thresholds: cross-fitted)", color=muted, fontsize=9)
    a1.set_title(f"Top {len(top)} prediction sets", loc="left", color=ink, fontsize=11)
    names = {"single": "single run", "ensemble": "your ensemble", "greedy": "greedy ensemble"}
    kinds = [k for k in KIND_COLORS if any(r["kind"] == k for r in top)]
    a1.legend([plt.Rectangle((0, 0), 1, 1, color=KIND_COLORS[k]) for k in kinds], [names[k] for k in kinds],
              loc="lower right", frameon=False, fontsize=8)
    series = [("tentative_f1", "macro-F1 (primary)", "#2a78d6"), ("micro_f1", "micro-F1", "#eb6834"),
              ("subset_acc", "subset accuracy", "#1baf7a")]
    yy = np.arange(len(groups)); h = 0.26
    for k, (key, lab, col) in enumerate(series):
        a2.barh(yy + (1 - k) * h, [r[key] or 0 for r in groups], height=h, color=col, label=lab, edgecolor="#fcfcfb", linewidth=1.5)
    for yi, r in zip(yy, groups):
        a2.text(r["tentative_f1"] + 0.003, yi + h, f"{r['tentative_f1']:.3f}", va="center", fontsize=7.5, color=muted)
    a2.set_yticks(yy, [r["members"][:48] for r in groups], fontsize=7.5, color=ink)
    a2.set_xlim(0, min(1.0, max(max(r[k] or 0 for r in groups) for k, _, _ in series) * 1.12))
    a2.set_xlabel("dev scores of the best set of each run / ensemble", color=muted, fontsize=9)
    a2.set_title("Best per run: macro-F1, micro-F1, subset accuracy", loc="left", color=ink, fontsize=11)
    a2.legend(loc="lower right", frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130, facecolor=fig.get_facecolor())
    plt.close(fig)
    print("chart saved:", path)

In [ ]:
%%writefile evaluate_phase5.py
#!/usr/bin/env python3
"""Local evaluator for Phase 5.

Truth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]
Prediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]
"""
import argparse,json,sys
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}; max_label=-1
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or not isinstance(r['gold'],list): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        seen=set()
        for label in r['gold']:
            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f'Invalid/duplicate label in truth item {n}.')
            seen.add(label); max_label=max(max_label,label)
        out[r['id']]=r['gold']
    return out,max_label+1

def read_pred(path,nlab):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or not isinstance(r['pred'],list) or r['id'] in out: raise ValueError(f'Invalid or duplicate prediction item {n}.')
        seen=set()
        for x in r['pred']:
            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f'Invalid/duplicate label in prediction item {n}.')
            seen.add(x)
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 5 multilabel predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    print(f'Examples: {len(t)}'); print(f'Labels: {n}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}')
    micro=f1_score(yt,yp,average='micro',zero_division=0)
    subset=accuracy_score(yt,yp)
    print(f'Micro-F1: {micro:.6f}')
    print(f'Subset Accuracy: {subset:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)

## N7b. Pre-flight check — every input present, distillation loss written? (stops here if not)

In [ ]:
# N7b — PRE-FLIGHT: checks every input this session needs and STOPS here (within a minute, not after hours) if one
# is missing. Read the printout: every line must show a path, not MISSING.
import json, glob, os, importlib, torch
import ls_common as L
C = json.load(open("/kaggle/working/run_config_ls.json"))
problems = []
print("tokenizer:", L.find_input("bpe_tok.py"), "|", L.find_input("bpe.json"))
archs = set()
for e in SESSION_EXPERIMENTS:
    x = C["experiments"][e]
    archs.add(x["arch"])
    if x.get("teacher"):
        archs.add(C["experiments"][x["teacher"]]["arch"])        # the teacher is rebuilt from its Phase-1 model too
for arch in sorted(archs):
    pat = C["checkpoints"][arch]
    hits = sorted(glob.glob(f"{L.INPUT_ROOT}/**/{pat}", recursive=True))
    print(f"Phase-1 checkpoint {arch:8s} ({pat}): {hits[0] if hits else 'MISSING'}")
    if not hits:
        problems.append(f"the Phase-1 '{arch}' checkpoint '{pat}' is not in any input — add the notebook / dataset "
                        f"that contains it" + (" (decoder: notebook27acca3fc3 = notebook D)" if arch == "decoder" else
                                               " (moe: phase1-train-extra / phase1-training2)" if arch == "moe" else ""))
for e in SESSION_EXPERIMENTS:
    t = C["experiments"][e].get("teacher")
    if t and t not in SESSION_EXPERIMENTS:
        f = L.find_runs("best.pt").get(t)
        print(f"distillation teacher '{t}' for {e}: {f + '/best.pt' if f else 'MISSING'}")
        if not f:
            problems.append(f"the trained teacher '{t}' (ls_runs/{t}/best.pt) was not found — train it in this session "
                            f"(add it to SESSION_EXPERIMENTS) or add the output of the notebook that trained it")
if any(C["experiments"][e].get("teacher") for e in SESSION_EXPERIMENTS):
    import ls_methods as M
    importlib.reload(M)
    try:
        z = torch.randn(3, 5); y = (torch.rand(3, 5) > 0.5).float()
        v = M.kd_loss_multilabel(z, torch.randn(3, 5), y, temperature=2.0, alpha=0.5)
        assert v.dim() == 0 and torch.isfinite(v), "kd_loss_multilabel must return one finite number"
        print(f"kd_loss_multilabel: OK (test value {v.item():.4f})")
    except NotImplementedError:
        problems.append("kd_loss_multilabel in N6 (section 7) is not written yet — write it, or take the distillation "
                        "run out of SESSION_EXPERIMENTS")
earlier = L.find_runs("test_scores.pkl")
print("finished runs found in the inputs (they join the lab):", sorted(earlier) or "none")
if problems:
    raise RuntimeError("PRE-FLIGHT FAILED — nothing was trained:\n  - " + "\n  - ".join(problems))
print("\nPRE-FLIGHT OK")

## N8. Data check + one-time tokenisation (sizes, statute statistics, document lengths, checkpoints)

In [ ]:
# N8 — read the data, tokenise every document ONCE (all CPU cores; cached in /kaggle/working/p5cache), then show sizes,
# statute statistics and document lengths. If this cell fails, the data format differs from the brief: paste the error.
import json, time, importlib, glob, os, shutil, numpy as np, torch
import ls_common as L
importlib.reload(L)
prev = sorted(glob.glob(f"{L.INPUT_ROOT}/**/p5cache/test_meta.json", recursive=True))
if prev and not os.path.exists(L.CACHE):                 # an earlier run's cache: no need to tokenise again
    shutil.copytree(os.path.dirname(prev[0]), L.CACHE)
    print("reusing the tokenisation cache from", os.path.dirname(prev[0]))
t0 = time.time()
L.build_cache()
print(f"tokenised in {time.time() - t0:.0f}s")
C = json.load(open("/kaggle/working/run_config_ls.json"))
win = C["defaults"]["window"]
for name in L.SPLITS:
    s = L.Split(name)
    n = s.lengths()
    lab = [x for x in s.labels if x is not None]
    info = (f"statutes per doc: mean {np.mean([len(x) for x in lab]):.2f}, max {max(len(x) for x in lab)}, "
            f"{sum(1 for x in lab if not x)} docs with none" if lab else "no labels")
    print(f"{name:5s}: {len(s):6d} docs | tokens per doc median {int(np.median(n))}, 90% {int(np.percentile(n, 90))}, "
          f"max {n.max()} | windows of {win}: median {int(np.median(np.ceil(n / win)))}, 90% "
          f"{int(np.percentile(np.ceil(n / win), 90))} | {info}")
K = L.n_labels()
tr, dv = L.Split("train"), L.Split("dev")
freq = np.bincount([x for ls in tr.labels for x in ls], minlength=K)
dfreq = np.bincount([x for ls in dv.labels for x in (ls or [])], minlength=K)
print(f"\nhead size K = {K} (largest statute id in train + dev + 1) | ids used in train: {(freq > 0).sum()}, in dev: {(dfreq > 0).sum()}")
print(f"largest id: train {max(max(l) for l in tr.labels if l)}, dev {max(max(l) for l in dv.labels if l)}  "
      "(the official evaluator rejects ids above the largest id of its reference — see MAX_LABEL in N3)")
used = freq[freq > 0]
print(f"training documents per statute: min {used.min()}, median {int(np.median(used))}, max {used.max()} | "
      f"statutes with < 10 docs: {(used < 10).sum()}, < 50: {(used < 50).sum()}")
print("ids in dev never seen in train:", [int(i) for i in np.flatnonzero((dfreq > 0) & (freq == 0))] or "none")
print("ids never used (gaps):", [int(i) for i in np.flatnonzero((freq == 0) & (dfreq == 0))] or "none")
test = L.Split("test")
assert len(set(test.ids)) == len(test.ids), "test document ids repeat"
print(f"dev: {len(dv.ids) - len(set(dv.ids))} repeated ids (each id is scored once, like the official evaluator)")
print("example test id:", test.ids[0])
for arch in sorted({C["experiments"][e]["arch"] for e in SESSION_EXPERIMENTS}):
    try:
        path = L.find_input(C["checkpoints"][arch])
        ck = torch.load(path, map_location="cpu", weights_only=False)
        print(f"checkpoint {arch:9s}: {path} | val_loss {ck.get('val_loss', float('nan')):.3f}")
    except Exception as e:
        print(f"checkpoint {arch:9s}: MISSING -> {e}")

## N9. CPU smoke test of the session's experiments

In [ ]:
# N9 — CPU smoke test (tiny random models, a few documents): every experiment must end with exit code 0.
import json, subprocess, sys
C = json.load(open("/kaggle/working/run_config_ls.json"))
smoke = []
for exp in SMOKE_EXPERIMENTS:
    t = C["experiments"][exp].get("teacher")
    smoke += [x for x in ([t] if t else []) + [exp] if x not in smoke]
for exp in smoke:
    r = subprocess.run([sys.executable, "train_ls.py", "--exp", exp, "--device", "cpu", "--smoke"],
                       capture_output=True, text=True, cwd="/kaggle/working")
    lines = [l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn(" not in l]
    print(f"===== {exp}: exit code {r.returncode}")
    print("\n".join(lines[-2:] if r.returncode == 0 else lines[-25:]))

## N10. Train the session's experiments on the GPUs + what each reached on its own

In [ ]:
# N10 — train SESSION_EXPERIMENTS: one experiment per GPU at a time; a distillation student waits for its teacher,
# a train_on_all run for its epochs_from run. Progress every 5 minutes.
import subprocess, sys, time, os, json, torch
C = json.load(open("/kaggle/working/run_config_ls.json"))
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
pending, running, finished = list(SESSION_EXPERIMENTS), {}, {}

def ready(exp):
    e = C["experiments"][exp]
    waits = [x for x in (e.get("teacher"), e.get("epochs_from")) if x]
    return all(t not in pending and t not in [r for r, _ in running.values()] for t in waits)

last_print = 0
while pending or running:
    for dev in devices:
        if dev in running and running[dev][1].poll() is not None:
            exp, p = running.pop(dev); finished[exp] = p.returncode
            print(time.strftime("%H:%M"), f"| {exp} finished on {dev} with exit code {p.returncode}", flush=True)
        if dev not in running:
            nxt = next((e for e in pending if ready(e)), None)
            if nxt:
                pending.remove(nxt)
                out = open(f"/kaggle/working/{nxt}.out", "w")
                running[dev] = (nxt, subprocess.Popen([sys.executable, "train_ls.py", "--exp", nxt, "--device", dev],
                                                      stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working"))
                print(time.strftime("%H:%M"), f"| started {nxt} on {dev}", flush=True)
    if time.time() - last_print > 300 and running:
        for dev, (exp, _) in running.items():
            lines = [l for l in open(f"/kaggle/working/{exp}.out").read().splitlines() if l.startswith(exp)]
            print(time.strftime("%H:%M"), "|", (lines[-1] if lines else f"{exp}: starting...")[:230], flush=True)
        last_print = time.time()
    time.sleep(15)
print("\nexit codes:", finished or "nothing trained this session")
for exp, code_ in finished.items():
    if code_ != 0:
        print(f"\n===== {exp} FAILED — last lines:\n" + open(f"/kaggle/working/{exp}.out").read()[-3000:])

In [ ]:
# N10b — what each run reached on its own (best checkpoint, all windows, its own doc_pool, one threshold for all)
import json, os
import ls_common as L
rows = [json.load(open(f"{f}/summary.json")) for e, f in L.find_runs("summary.json").items()
        if os.path.exists(f"{f}/test_scores.pkl")]          # this session's runs + finished earlier ones
print(f"{'run':22s} {'arch':9s} {'method':8s} {'train%':>7s} {'macroF1':>7s} {'microF1':>7s} {'subset':>6s} {'thr':>4s} "
      f"{'epochs':>7s} {'min':>5s} {'GPU GB':>6s} {'test s':>6s}")
for s in sorted(rows, key=lambda s: -(s.get("dev_macro_f1") or -1)):
    f = lambda k: f"{s[k]:.3f}" if s.get(k) is not None else "  n/a"
    print(f"{s['exp']:22s} {s['arch']:9s} {s['method']:8s} {100 * s['params_trainable'] / s['params_total']:7.2f} "
          f"{f('dev_macro_f1'):>7s} {f('dev_micro_f1'):>7s} {f('dev_subset_acc'):>6s} {str(s.get('dev_threshold', '')):>4s} "
          f"{s['epochs_done']:7.2f} {s['train_minutes']:5.0f} {str(s['peak_gpu_gb']):>6s} {s['test_predict_sec']:6.0f}")

## N11. The lab — prediction sets, zips, index.csv, score chart

In [ ]:
# N11 — the lab (CPU, several minutes): every finished run x SELECTS x WINDOW_AGGS x RULES x MIN1; your
# PROB_ENSEMBLES; greedy ensembles. Writes zips, index.csv and the score chart.
import importlib, ls_common as L, ls_lab
importlib.reload(L); importlib.reload(ls_lab)
rules = ls_lab.parse_rules(RULES)
lab = ls_lab.Lab(max_label=MAX_LABEL)
lab.load_runs(RUN_DIR_OVERRIDES)
lab.singles(SELECTS, WINDOW_AGGS, rules, MIN1)
lab.ensembles(PROB_ENSEMBLES, SELECTS, WINDOW_AGGS, rules, MIN1)
if GREEDY:
    lab.greedy(SELECTS, WINDOW_AGGS, rules, MIN1)
rows = lab.finish(TOP_N_ZIPS)
ls_lab.table(rows, 40)
ls_lab.chart(rows, "/kaggle/working/submissions/score_chart.png")
from IPython.display import Image, display
display(Image("/kaggle/working/submissions/score_chart.png"))

## N12. Check the zips + confirm the lab's scores with the official evaluator

In [ ]:
# N12 — check every zip (format, ids, statute ids) and confirm the lab's tentative scores with the official evaluator
# (evaluate_phase5.py) on the 5 best written prediction sets.
import zipfile, json, glob, csv, subprocess, sys
import ls_common as L
test_ids = L.Split("test").ids
K = L.n_labels()
zips = sorted(glob.glob("/kaggle/working/submissions/*.zip")) + sorted(glob.glob("/kaggle/working/ls_runs/*/submission_*.zip"))
bad_zips = 0
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    ids = [r["id"] for r in pred]
    ok = (names == ["prediction.json"] and isinstance(pred, list) and set(ids) == set(test_ids) and len(ids) == len(test_ids)
          and all(isinstance(r["pred"], list) and len(set(r["pred"])) == len(r["pred"])
                  and all(type(x) is int and 0 <= x < K for x in r["pred"]) for r in pred))
    bad_zips += not ok
    if not ok:
        print("PROBLEM", z)
print(f"{len(zips)} zips checked, {bad_zips} with problems")
V = "/kaggle/working/submissions/dev_predictions"
rows = [r for r in csv.DictReader(open("/kaggle/working/submissions/index.csv")) if r["file"].endswith(".zip") and r["tentative_f1"]]
for r in rows[:5]:
    out = subprocess.run([sys.executable, "evaluate_phase5.py", "--truth", f"{V}/dev_truth.json",
                          "--prediction", f"{V}/{r['name']}.json"], capture_output=True, text=True)
    off = [l.split()[-1] for l in out.stdout.splitlines() if l.startswith("Macro-F1")] or [out.stderr.strip()[-200:]]
    print(f"{r['name'][:64]:64s} lab {r['tentative_f1']} | official {off[0]}")

## N13. Error analysis of one prediction set (rare statutes, missed / over-predicted statutes, set sizes)

In [ ]:
# N13 — error analysis of one written prediction set (material for the report): F1 by how often a statute occurs in
# training, the statutes most often missed / wrongly added, and the statute most often present when one is wrongly added.
import json, csv, numpy as np
import ls_common as L
NAME = None          # <- a name from submissions/index.csv; None = the best written one
V = "/kaggle/working/submissions/dev_predictions"
if NAME is None:
    NAME = next(r["name"] for r in csv.DictReader(open("/kaggle/working/submissions/index.csv"))
                if r["file"].endswith(".zip") and r["tentative_f1"])
print("analysing:", NAME)
truth = {r["id"]: r["gold"] for r in json.load(open(f"{V}/dev_truth.json"))}
pred = {r["id"]: r["pred"] for r in json.load(open(f"{V}/{NAME}.json"))}
ids = list(truth)
n = max(max(g) for g in truth.values()) + 1
G = L.label_matrix([truth[i] for i in ids], n); P = L.label_matrix([pred[i] for i in ids], n)
f1 = L.per_label_f1(G, P)
freq = np.bincount([x for ls in L.Split("train").labels for x in ls], minlength=n)[:n]
print(f"macro-F1 {f1.mean():.4f} over {n} statute ids | gold statutes per doc {G.sum(1).mean():.2f}, predicted {P.sum(1).mean():.2f}")
print("\nF1 by training frequency of the statute:")
for a, b in [(0, 0), (1, 9), (10, 49), (50, 199), (200, 999), (1000, 10 ** 9)]:
    m = (freq >= a) & (freq <= b)
    if m.any():
        print(f"  {a:5d} - {b if b < 10 ** 9 else 'max':>5}: {m.sum():4d} statutes, mean F1 {f1[m].mean():.3f}, "
              f"dev positives {G[:, m].sum():6d}")
fn = (G & ~P).sum(0); fp = (~G & P).sum(0)
print("\nmost often MISSED (false negatives):     " + ", ".join(f"{k} ({fn[k]}, F1 {f1[k]:.2f})" for k in np.argsort(-fn)[:10]))
print("most often WRONGLY ADDED (false positives): " + ", ".join(f"{k} ({fp[k]}, F1 {f1[k]:.2f})" for k in np.argsort(-fp)[:10]))
print("\nwrongly added statute -> the gold statute most often present in those documents (label co-occurrence):")
for k in np.argsort(-fp)[:8]:
    rows_ = (~G[:, k]) & P[:, k]
    co = G[rows_].sum(0)
    if co.any():
        print(f"  {k:4d} wrongly added in {rows_.sum():4d} docs; most common gold there: {int(co.argmax())} ({co.max()} docs)")
exact = (G == P).all(1)
print(f"\nsubset accuracy {exact.mean():.3f} | by number of gold statutes:")
for c in range(1, int(G.sum(1).max()) + 1):
    m = G.sum(1) == c
    if m.sum() >= 5:
        print(f"  {c} gold statute(s): {m.sum():5d} docs, exact {exact[m].mean():.3f}, mean predicted {P[m].sum(1).mean():.2f}")